# Практическое занятие 14. Визуализация данных: построение графиков, диаграмм и создание облака слов (Word Cloud)

**Технологии больших данных (Big Data)**, 6 семестр.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/notebooks/p14.ipynb)

**Цели занятия:**

- Выбирать тип графика под задачу: сравнение, динамика, распределение, доли, связь величин
- Строить графики в matplotlib (bar, line, hist, pie, scatter) с заголовками и подписями осей, объединять их через `subplots`
- Быстро строить графики методом pandas `.plot` и статистические диаграммы seaborn
- Создавать облако слов (Word Cloud) по частотам корпуса с учётом стоп-слов и апострофов

> Автоматическая проверка заданий доступна в веб-тренажёре. В блокноте выполняйте код и сверяйте результат с условием.

In [ ]:
# ▶ Подготовка среды: выполните эту ячейку первой.
# Загружает учебные данные из репозитория тренажёра и запускает локальный веб-сервер
# (нужен для заданий с requests: учебный сайт SITE и учебный API).
import os, sys, subprocess, time

REPO = "https://github.com/OWNER/REPO.git"
if not os.path.exists("data/manifest.json"):
    if "OWNER/REPO" in REPO:
        raise SystemExit("Укажите в REPO адрес репозитория тренажёра на GitHub, например https://github.com/<логин>/bigdata-trainer.git")
    if not os.path.exists("bigdata-trainer"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "bigdata-trainer"], check=True)
    os.chdir("bigdata-trainer")

subprocess.Popen([sys.executable, "-m", "http.server", "8765"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
BASE_URL = "http://localhost:8765/"
SITE = BASE_URL + "data/site/"
API = BASE_URL + "data/api/v1/"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl", "wordcloud", "seaborn"])
print("Готово. Рабочая папка:", os.getcwd())

## Теория

## Зачем визуализировать данные

Таблица из тысяч строк ничего не говорит человеку, а хороший график показывает закономерность за секунды: пик активности в соцсети, самый популярный факультет, связь посещаемости и оценок. В Big Data визуализация — последний шаг конвейера: сначала данные **агрегируются** (группировка, подсчёт частот), а на график выводится уже компактный результат.

## Анатомия графика matplotlib

**Figure** — весь рисунок, **Axes** — отдельная область с осями (на одном рисунке их может быть несколько).

```python
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))     # рисунок 8×4 дюйма с одной областью
ax.bar(["A", "B", "C"], [5, 3, 7], color="steelblue")
ax.set_title("Заголовок графика")
ax.set_xlabel("Подпись оси X")
ax.set_ylabel("Подпись оси Y (единицы!)")
plt.show()                                  # показать рисунок
```

| Задача | Тип графика | matplotlib |
|---|---|---|
| Сравнить категории | столбчатая | `ax.bar`, `ax.barh` |
| Показать динамику во времени | линейный | `ax.plot(x, y, marker="o")` |
| Увидеть распределение величины | гистограмма | `ax.hist(x, bins=15)` |
| Показать доли целого (до 5–6 частей) | круговая | `ax.pie(values, labels=..., autopct="%1.0f%%")` |
| Найти связь двух величин | точечная | `ax.scatter(x, y, alpha=0.6)` |

## Несколько графиков на одном рисунке

```python
fig, axes = plt.subplots(1, 3, figsize=(15, 4))   # 1 строка, 3 столбца
axes[0].hist(df["gpa"], bins=15)
axes[1].scatter(df["attendance"], df["gpa"], alpha=0.5)
axes[2].pie(levels, labels=levels.index, autopct="%1.0f%%")
fig.suptitle("Общий заголовок")
plt.tight_layout()                                 # убрать наложения подписей
```

Для сетки `plt.subplots(2, 2)` области адресуются двумя индексами: `axes[0, 1]` — первая строка, второй столбец.

## Быстрые графики pandas

У Series и DataFrame есть метод `.plot` — обёртка над matplotlib:

```python
df["city"].value_counts().head(5).plot(kind="barh", title="Топ-5 городов")
table.plot(ax=axes[1], marker="o")      # DataFrame: каждая колонка — отдельная линия с легендой
series.plot(kind="pie", autopct="%1.0f%%")
```

Параметр `ax=` позволяет нарисовать график pandas в нужной области `subplots`.

## seaborn — статистические графики

seaborn строит графики прямо по DataFrame и сам считает средние, квартили, плотности:

```python
import seaborn as sns
sns.histplot(data=df, x="gpa", hue="gender", kde=True)    # гистограмма + плотность
sns.boxplot(data=df, x="faculty", y="gpa")                # медиана, квартили, выбросы
sns.barplot(data=df, x="english_level", y="attendance")   # среднее и доверительный интервал
sns.heatmap(df[cols].corr(), annot=True, cmap="coolwarm") # матрица цветом
```

> **Заметка: «Ящик с усами» (boxplot)**  
> Линия внутри ящика — **медиана**, границы ящика — 1-й и 3-й квартили, «усы» — разброс без выбросов, отдельные точки — выбросы.

## Облако слов (Word Cloud)

Облако слов показывает частые слова корпуса: чем чаще слово, тем крупнее шрифт. Это не точный график, а наглядная иллюстрация темы текста.

```python
from wordcloud import WordCloud
from collections import Counter

freq = Counter(words)                         # слова уже очищены от стоп-слов
wc = WordCloud(width=600, height=300, background_color="white",
               max_words=100).generate_from_frequencies(freq)
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.show()
wc.words_                                     # {слово: относительный размер}
```

- **Стоп-слова** (va, bu, bilan, uchun…) обязательно удаляют: иначе они займут всё облако.
- Метод `generate(text)` сам разбивает текст и удаляет **английские** стоп-слова; для узбекского надёжнее считать частоты самим и вызывать `generate_from_frequencies`.

> **Внимание: Апострофы и шрифт**  
> Встроенный шрифт WordCloud не содержит символов ʻ (U+02BB) и ʼ (U+02BC) — вместо них появятся пустые квадраты. Перед построением облака замените их на обычный `'`: `w.replace("ʻ", "'").replace("ʼ", "'")` — или укажите шрифт с нужными глифами: `WordCloud(font_path="arial.ttf")`.

## Правила хорошего графика

1. Заголовок и подписи осей с единицами измерения (сум, млн сум, %, шт.).
2. Столбцы сортируют по величине; длинные подписи — горизонтальная диаграмма `barh`.
3. Круговая диаграмма — только для 2–6 долей одного целого.
4. Одна мысль — один график; для сравнения нескольких — `subplots` с общей шкалой.
5. Сохранение для отчёта: `fig.savefig("chart.png", dpi=200, bbox_inches="tight")`.

## Задание 1. Столбчатая диаграмма

Постройте **вертикальную** столбчатую диаграмму числа студентов по факультетам (`data/students.csv`).

- `counts` — Series: число студентов на каждом факультете (`value_counts`);
- график `ax.bar` с заголовком и подписями обеих осей.

<details><summary>💡 Подсказка</summary>

`counts = df["faculty"].value_counts()`; затем `ax.set_title("Студенты по факультетам")`, `ax.set_xlabel("Факультет")`, `ax.set_ylabel("Число студентов")`.

</details>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/students.csv")
counts = ...      # число студентов по факультетам

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(counts.index, counts.values, color="steelblue")
# добавьте заголовок и подписи осей

plt.xticks(rotation=15)
plt.show()

## Задание 2. Линейный график: активность в соцсетях

Файл `data/tweets_stream.csv` — 30 минут постов в соцсетях (21.03.2026, Навруз). Постройте **линейный** график числа постов по минутам.

1. Удалите дубликаты строк.
2. `per_min` — Series: число постов за каждую минуту (индекс — время, по возрастанию). Подойдёт `tw["timestamp"].dt.floor("min").value_counts().sort_index()` или `resample("1min")`.
3. `peak` — минута с максимальным числом постов.
4. Линейный график с маркерами, заголовком и подписями осей.

<details><summary>💡 Подсказка</summary>

`tw = tw.drop_duplicates()`; `per_min = tw["timestamp"].dt.floor("min").value_counts().sort_index()`; `peak = per_min.idxmax()`; `ax.plot(per_min.index, per_min.values, marker="o")`.

</details>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

tw = pd.read_csv("data/tweets_stream.csv", parse_dates=["timestamp"])
print("Всего строк:", len(tw))

tw = ...           # без дубликатов
per_min = ...      # постов в минуту
peak = ...         # минута-пик

fig, ax = plt.subplots(figsize=(10, 4))
# ax.plot(...)

plt.show()
print("Пик активности:", peak)

## Задание 3. Несколько графиков: hist, scatter, pie

Соберите «панель» из трёх графиков по `data/students.csv` на одном рисунке `plt.subplots(1, 3)`:

1. `axes[0]` — гистограмма GPA из **15** интервалов (уже готова);
2. `axes[1]` — точечная диаграмма: по оси X посещаемость `attendance`, по оси Y — `gpa`; подпишите оси;
3. `axes[2]` — круговая диаграмма долей уровней английского `levels` с процентами (`autopct="%1.0f%%"`).

У каждого графика должен быть заголовок.

<details><summary>💡 Подсказка</summary>

`axes[1].scatter(df["attendance"], df["gpa"], alpha=0.5)`; `axes[2].pie(levels, labels=levels.index, autopct="%1.0f%%")`; заголовки — `axes[i].set_title(...)`.

</details>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/students.csv")
levels = df["english_level"].value_counts().sort_index()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df["gpa"], bins=15, color="teal", edgecolor="white")
axes[0].set_title("Распределение GPA")
axes[0].set_xlabel("GPA")
axes[0].set_ylabel("Студентов")

# axes[1]: точечная диаграмма attendance — gpa

# axes[2]: круговая диаграмма levels

fig.suptitle("Студенты: успеваемость и посещаемость")
plt.tight_layout()
plt.show()

## Задание 4. Графики pandas: продажи книг

Продажи книжного магазина (`data/sales.xlsx`, листы 2025 и 2026) уже объединены в `sales`, добавлены `revenue` (сум), `year` и `month`.

1. `rev_by_cat` — выручка по категориям в **млн сум**, отсортированная **по возрастанию** (тогда на горизонтальной диаграмме крупнейшая категория окажется вверху).
2. `by_month` — таблица выручки (млн сум): строки — месяцы 1…12, столбцы — годы (`pivot_table`).
3. Нарисуйте методом pandas `.plot` на одном рисунке 1×2: слева `rev_by_cat` (`kind="barh"`), справа `by_month` — две линии с маркерами и легендой. Дайте графикам заголовки.

<details><summary>💡 Подсказка</summary>

`rev_by_cat = (sales.groupby("category")["revenue"].sum() / 1e6).sort_values()`; `by_month = sales.pivot_table(index="month", columns="year", values="revenue", aggfunc="sum") / 1e6`.

</details>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

books = pd.read_excel("data/sales.xlsx", sheet_name=None)
sales = pd.concat([books["2025"], books["2026"]], ignore_index=True)
sales["revenue"] = sales["qty"] * sales["price_uzs"]
sales["year"] = sales["date"].dt.year
sales["month"] = sales["date"].dt.month

rev_by_cat = ...
by_month = ...

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
# rev_by_cat.plot(kind="barh", ax=axes[0], ...)
# by_month.plot(ax=axes[1], marker="o", ...)
plt.tight_layout()
plt.show()

## Задание 5. seaborn: четыре статистических графика

Постройте по `data/students.csv` рисунок 2×2 средствами **seaborn**:

| Область | График |
|---|---|
| `axes[0, 0]` | `sns.histplot`: GPA с разбивкой по полу и кривой плотности (готово) |
| `axes[0, 1]` | `sns.boxplot`: GPA по факультетам |
| `axes[1, 0]` | `sns.barplot`: средняя посещаемость по уровням английского (`order=["A2", "B1", "B2", "C1"]`) |
| `axes[1, 1]` | `sns.heatmap`: матрица корреляций `corr` с подписями значений (`annot=True`) |

`corr` — корреляции между столбцами `age`, `course`, `gpa`, `attendance`. Каждому графику дайте заголовок.

<details><summary>💡 Подсказка</summary>

`corr = df[["age", "course", "gpa", "attendance"]].corr()`; `sns.boxplot(data=df, x="faculty", y="gpa", ax=axes[0, 1])`; `sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", ax=axes[1, 1])`.

</details>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("data/students.csv")
corr = ...

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

sns.histplot(data=df, x="gpa", hue="gender", kde=True, ax=axes[0, 0])
axes[0, 0].set_title("GPA: юноши и девушки")

# axes[0, 1]: boxplot
# axes[1, 0]: barplot
# axes[1, 1]: heatmap

plt.tight_layout()
plt.show()

## Задание 6. Облако слов по корпусу

Подготовлены: `texts` — 16 новостей на узбекском (`data/corpus_uz`, апострофы разные), `stop` — стоп-слова, функция `normalize(text)` (нижний регистр + канонические апострофы ʻ и ʼ).

1. `tokens` — все слова корпуса: `re.findall(r"\w+", normalize(text))`, только буквенные (`w.isalpha()`), длиной не меньше 3, не входящие в `stop`.
2. `freq = Counter(tokens)`; выведите 10 самых частых слов.
3. `freq_ascii` — те же частоты, но в словах ʻ и ʼ заменены на обычный апостроф `'` (шрифт WordCloud не знает ʻ ʼ).
4. `wc = WordCloud(width=600, height=300, background_color="white", max_words=100).generate_from_frequencies(freq_ascii)`; покажите облако `plt.imshow(wc)` без осей и с заголовком.

<details><summary>💡 Подсказка</summary>

Условие отбора: `if w.isalpha() and len(w) >= 3 and w not in stop: tokens.append(w)`. Замена апострофов: `freq_ascii[w.replace("ʻ", "'").replace("ʼ", "'")] += c`. Показ: `plt.imshow(wc, interpolation="bilinear")`, `plt.axis("off")`, `plt.title(...)`.

</details>

In [ ]:
# Подготовительный код задания
import re, glob
from collections import Counter

def normalize(text):
    """Нижний регистр и единые апострофы: oʻ/gʻ (U+02BB), остальные — ʼ (U+02BC)."""
    text = text.lower()
    for ch in ["'", "\u2018", "\u2019", "\x60", "\u02bb"]:
        text = text.replace(ch, "\u02bc")
    return re.sub("([og])\u02bc", "\\1\u02bb", text)

texts = [open(f, encoding="utf-8").read() for f in sorted(glob.glob("data/corpus_uz/uz_*.txt"))]
stop = set(open("data/stopwords_uz.txt", encoding="utf-8").read().split())

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

tokens = []
for text in texts:
    for w in re.findall(r"\w+", normalize(text)):
        ...   # добавьте w в tokens, если оно подходит

freq = Counter(tokens)
print(freq.most_common(10))

freq_ascii = Counter()
for w, c in freq.items():
    ...

wc = ...

plt.figure(figsize=(10, 5))
# plt.imshow(...)
plt.show()

## Самостоятельная работа

## Самостоятельная работа: «Анализ больших объёмов данных, формируемых в социальных сетях»

По файлу `data/tweets_stream.csv` подготовьте мини-дашборд (рисунок 2×2) и короткий вывод (5–7 предложений):

1. Доля постов по платформам (столбчатая или круговая диаграмма).
2. Тепловая карта «платформа × язык поста» (`pd.crosstab` + `sns.heatmap`).
3. Топ-10 хештегов (`re.findall(r"#\w+", text)`) — горизонтальная диаграмма.
4. Облако слов узбекоязычных постов (`lang == "uz"`) без стоп-слов.

Дополнительно: сравните среднее число лайков у постов с изображением и без (`has_image`) с помощью `sns.boxplot`.